In [ ]:
from dipy.reconst import axdki as dki
import numpy as np                                                                                        

from dipy.core.gradients import gradient_table
from dipy.data import get_fnames
from dipy.denoise.localpca import mppca
from dipy.io.gradients import read_bvals_bvecs
from dipy.io.image import load_nifti
from dipy.reconst.weights_method import simple_cutoff, weights_method_wls_m_est
from dipy.segment.mask import median_otsu

fraw, fbval, fbvec, t1_fname = get_fnames(name="cfin_multib")

data, affine = load_nifti(fraw)
bvals, bvecs = read_bvals_bvecs(fbval, fbvec)
gtab = gradient_table(bvals, bvecs=bvecs)
bval_sel = np.zeros_like(gtab.bvals)
bval_sel[bvals == 0] = 1
bval_sel[bvals == 600] = 1
bval_sel[bvals == 1000] = 1
bval_sel[bvals == 2000] = 1

data = data[..., bval_sel == 1]
gtab = gradient_table(bvals[bval_sel == 1], bvecs=bvecs[bval_sel == 1])
datamask, mask = median_otsu(data, vol_idx=[0, 1], median_radius=4, numpass=2, dilate=1)
dkimodel = dki.AxialSymmetricDiffusionKurtosisModel(datamask,gtab)
result = dkimodel.fit()
